In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
DF = pd.read_excel('PQM dataset.xlsx', sheet_name='Frequency_1Min')
print("DataFrame shape:", DF.shape)

In [ ]:
# calculating z-scores for outlier detection
def calc_zscore(column):
    mean = np.mean(column)
    std_dev = np.std(column)

    if std_dev == 0:
        return np.zeros_like(column)
    
    z_scores = (column - mean) / std_dev
    return z_scores

In [ ]:
columns = [col for col in DF.columns if col not in ['TimeID']]

z_scores = DF[columns].apply(calc_zscore)

print(z_scores.head())

In [ ]:
# cleaning data by removing outliers and filling NaN values
outlier_mask = (np.abs(z_scores) > 3).any(axis=1)
print(f"Found {outlier_mask.sum()} outliers in the dataset")

cleanDF = DF[~outlier_mask]
print("Cleaned DataFrame shape:", cleanDF.shape)
cleanDF = cleanDF.dropna()
print("Cleaned DataFrame shape after dropping NaNs:", cleanDF.shape)

In [ ]:
# window size = 6 for 12 columns
input_size = 3
power_columns = [col for col in cleanDF.columns if col != 'TimeID']

x_list = []
y_list = []

In [ ]:
for i in range(len(power_columns) - input_size):
    input_window = power_columns[i:i + input_size]
    target_column = power_columns[i + input_size]

    x_list.append(cleanDF[input_window].values)
    y_list.append(cleanDF[target_column].values)

In [ ]:
x = np.vstack(x_list)
y = np.hstack(y_list)

print("Input shape:", x.shape)
print("Target shape:", y.shape)

In [ ]:
split_index = int(0.8 * len(x))

x_train = x[:split_index]
y_train = y[:split_index]

x_test = x[split_index:]
y_test = y[split_index:]

print("Training set shape:", x_train.shape, y_train.shape)
print("Test set shape:", x_test.shape, y_test.shape)

In [ ]:
model = LinearRegression()
model.fit(x_train, y_train)

In [ ]:
# test train split
y_pred = model.predict(x_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Model Performance on Test Set:")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R-squared (R2): {r2:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(y_test[:200], label='Actual', color='blue')
plt.plot(y_pred[:200], label='Predicted', color='red')
plt.title('Actual vs Predicted Frequency (first 200 values)')
plt.xlabel('Time Step')
plt.ylabel('Frequency Value')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(y_test, label='Actual', color='blue')
plt.plot(y_pred, label='Predicted', color='red')
plt.title('Actual vs Predicted Frequency (all values)')
plt.xlabel('Time Step')
plt.ylabel('Frequency Value')
plt.legend()
plt.show()